# 🃏 Notebook 17 — The Hidden Board
### Counterfactual Regret Minimization (CFR) & Poker

**Series**: RL Notebook Series · Act VI — Grandmasters · Post 17 of 17

---

## The Blind Spot

In Notebook 16, AlphaZero conquered Tic-Tac-Toe (and by extension, chess and Go).
But all these games share one property: **perfect information** — both players see the entire board.

**Poker** breaks this completely:
- You can see your own cards but **not your opponent's**
- The optimal strategy is **mixed** (probabilistic) — you *must* randomize
- **Bluffing** is not a psychological trick but a mathematical necessity

### Why MCTS Fails

MCTS works by simulating future game states. But in poker, you don't know what cards
your opponent holds — you can't simulate a state you can't see. We need a fundamentally
different approach.

### The Solution: CFR

**Counterfactual Regret Minimization** (Zinkevich et al., 2007) iteratively improves
a strategy by minimizing *regret* — how much the player wishes they had played differently.
The remarkable result: the **average strategy** converges to a **Nash equilibrium**,
a strategy that **cannot be exploited** by any opponent.

Today we'll build CFR from scratch on **Kuhn Poker** — the simplest non-trivial poker game —
and watch **bluffing emerge naturally from the math**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import random

%matplotlib inline
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.grid': True, 'grid.alpha': 0.3})
np.random.seed(42)
random.seed(42)

## 1. Kuhn Poker — The Simplest Poker Game

**Kuhn Poker** (Harold Kuhn, 1950) is a toy poker game with only 3 cards and 2 players.
Despite its simplicity, it captures the essential challenge of imperfect information:
bluffing, calling, and folding all emerge naturally.

### Rules
- **Deck**: just 3 cards — Jack (J), Queen (Q), King (K)
- **Deal**: each player gets 1 card. The third card is unseen.
- **Ante**: both players put 1 chip in the pot (starting pot = 2)
- **Actions**: `bet` (add 1 chip) or `check`/`call`/`fold`

### Game Flow
```
Player 1 acts first:
  ├── check → Player 2 acts:
  │     ├── check → Showdown (higher card wins the pot)
  │     └── bet   → Player 1 acts:
  │           ├── call → Showdown (pot = 4)
  │           └── fold → Player 2 wins (pot = 3)
  └── bet   → Player 2 acts:
        ├── call → Showdown (pot = 4)
        └── fold → Player 1 wins (pot = 3)
```

### Card Rankings
K > Q > J — the higher card wins at showdown.

In [ ]:
CARDS = ['J', 'Q', 'K']
CARD_RANK = {'J': 0, 'Q': 1, 'K': 2}


class KuhnPoker:
    """Kuhn Poker game state.
    
    Information set string format: "card:history"
    Example: "Q:cb" means "I hold Queen, action history is check-then-bet"
    """
    def __init__(self):
        self.cards = None      # [player1_card, player2_card]
        self.history = ''      # Action history: sequence of 'b' (bet), 'c' (check), 'k' (call), 'f' (fold)
        self.current_player = 0  # 0 or 1
    
    def deal(self):
        """Shuffle and deal cards to both players."""
        deck = CARDS.copy()
        random.shuffle(deck)
        self.cards = [deck[0], deck[1]]
        self.history = ''
        self.current_player = 0
        return self
    
    def get_info_set(self):
        """Return the information set string for the current player.
        
        The info set captures everything the current player knows:
        their own card + the action history. They do NOT know the opponent's card.
        """
        return f"{self.cards[self.current_player]}:{self.history}"
    
    def get_valid_actions(self):
        """Return valid actions for the current player."""
        if self.history == '':
            return ['b', 'c']       # Player 1: bet or check
        elif self.history == 'c':
            return ['b', 'c']       # Player 2 after check: bet or check
        elif self.history == 'b':
            return ['k', 'f']       # Player 2 after bet: call or fold
        elif self.history == 'cb':
            return ['k', 'f']       # Player 1 after check-bet: call or fold
        return []
    
    def is_terminal(self):
        """Check if the game is over."""
        return self.history in ['cc', 'bc', 'bf', 'bk', 'cbk', 'cbf']
    
    def get_payoff(self):
        """Return payoff for Player 1 (zero-sum: Player 2 gets the negative).
        
        Payoffs are in terms of chips won/lost from the ante.
        """
        assert self.is_terminal(), "Game is not over!"
        
        # Someone folded
        if self.history == 'bf':    # P2 folded after P1 bet
            return +1               # P1 wins the pot (ante only)
        if self.history == 'cbf':   # P1 folded after P2 bet
            return -1               # P1 loses ante
        if self.history == 'bc':    # P2 folded (checked after bet = invalid, see valid_actions)
            return +1               # shouldn't happen with valid play
        
        # Showdown — compare cards
        p1_rank = CARD_RANK[self.cards[0]]
        p2_rank = CARD_RANK[self.cards[1]]
        winner = 1 if p1_rank > p2_rank else -1  # +1 if P1 wins, -1 if P2 wins
        
        if self.history in ['bk', 'cbk']:  # Someone called a bet
            return winner * 2               # Winner gets 2 (ante + bet)
        else:                                # Both checked: 'cc'
            return winner * 1               # Winner gets 1 (ante only)
    
    def play(self, action):
        """Apply an action and advance the game. Returns a new KuhnPoker state."""
        new = KuhnPoker()
        new.cards = self.cards
        new.history = self.history + action
        
        # Determine next player
        if not new.is_terminal():
            if len(new.history) % 2 == 0:
                new.current_player = 0
            else:
                new.current_player = 1
            # Special case: after 'cb', it's Player 1's turn again
            if new.history == 'cb':
                new.current_player = 0
        
        return new


# Quick test: play random games
def play_random_kuhn(verbose=False):
    game = KuhnPoker().deal()
    if verbose:
        print(f"Deal: P1={game.cards[0]}, P2={game.cards[1]}")
    while not game.is_terminal():
        actions = game.get_valid_actions()
        action = random.choice(actions)
        if verbose:
            info = game.get_info_set()
            print(f"  P{game.current_player+1} ({info}) plays '{action}'")
        game = game.play(action)
    payoff = game.get_payoff()
    if verbose:
        print(f"  Result: P1 {'wins' if payoff > 0 else 'loses'} {abs(payoff)} chip(s)")
    return payoff

print("5 random Kuhn Poker games:\n")
for i in range(5):
    print(f"Game {i+1}:")
    play_random_kuhn(verbose=True)
    print()

# Average payoff over many games
payoffs = [play_random_kuhn() for _ in range(10000)]
print(f"Average P1 payoff over 10,000 random games: {np.mean(payoffs):.4f}")

## 2. Information Sets — What You Know vs What Is

In **perfect-information** games (chess, Tic-Tac-Toe), each game state is unique and fully visible.
In **imperfect-information** games, multiple states look identical to a player — these form an
**information set**.

Example: You hold a Queen and the history is "check, bet" (info set `Q:cb`).
Your opponent could hold J or K — you don't know which. Both possibilities
are in the same information set. Your strategy must be the same for both.

This is why MCTS breaks: you can't simulate forward when you don't know which state you're in.

In [ ]:
# Enumerate all information sets in Kuhn Poker
def enumerate_info_sets():
    """Walk the game tree and collect all information sets."""
    info_sets = {}  # {info_set_string: {'player': int, 'actions': list}}
    
    def walk(game):
        if game.is_terminal():
            return
        info = game.get_info_set()
        actions = game.get_valid_actions()
        if info not in info_sets:
            info_sets[info] = {
                'player': game.current_player,
                'actions': actions,
            }
        for action in actions:
            walk(game.play(action))
    
    # Try all possible deals
    for c1 in CARDS:
        for c2 in CARDS:
            if c1 != c2:
                game = KuhnPoker()
                game.cards = [c1, c2]
                game.history = ''
                game.current_player = 0
                walk(game)
    
    return info_sets

info_sets = enumerate_info_sets()

print(f"Kuhn Poker has {len(info_sets)} information sets:\n")
print(f"{'Info Set':<10} {'Player':<8} {'Actions':<15}")
print("─" * 35)
for info, data in sorted(info_sets.items()):
    action_str = ', '.join(data['actions'])
    print(f"{info:<10} P{data['player']+1:<7} {action_str:<15}")

## 3. Regret and Regret Matching

The key idea behind CFR is **regret**: after each round, we ask
*"how much do I wish I had played a different action?"*

### Regret

The **regret** for not having played action $a$ at information set $I$ after $T$ rounds:

$$R^T(I, a) = \sum_{t=1}^{T} \left[ v_i(I, a) - v_i(I) \right]$$

Where $v_i(I, a)$ is the value of always playing action $a$ at $I$, and $v_i(I)$ is the value
of the current strategy.

### Regret Matching

**Regret matching** converts cumulative regrets into a strategy:

$$\sigma^{T+1}(I, a) = \begin{cases} \frac{\max(R^T(I,a), 0)}{\sum_{a'} \max(R^T(I,a'), 0)} & \text{if denominator} > 0 \\ \frac{1}{|A(I)|} & \text{otherwise (uniform)} \end{cases}$$

**Intuition**: play actions you **regret not having played** more often.
If all regrets are non-positive (you're happy with your choices), play uniformly.

In [ ]:
def regret_matching(regret_sum, actions):
    """Convert cumulative regrets into a strategy via regret matching.
    
    Args:
        regret_sum: dict {action: cumulative_regret}
        actions: list of valid actions
    
    Returns:
        strategy: dict {action: probability}
    """
    # Clip negative regrets to zero, then normalize
    positive_regrets = {a: max(regret_sum.get(a, 0), 0) for a in actions}
    total = sum(positive_regrets.values())
    
    if total > 0:
        return {a: positive_regrets[a] / total for a in actions}
    else:
        # All regrets non-positive → play uniformly
        return {a: 1.0 / len(actions) for a in actions}


# Quick test
print("Regret matching examples:")

r1 = {'b': 3.0, 'c': -1.0}
s1 = regret_matching(r1, ['b', 'c'])
print(f"  Regrets {r1} → Strategy {s1}")

r2 = {'b': 2.0, 'c': 2.0}
s2 = regret_matching(r2, ['b', 'c'])
print(f"  Regrets {r2} → Strategy {s2}")

r3 = {'b': -1.0, 'c': -2.0}
s3 = regret_matching(r3, ['b', 'c'])
print(f"  Regrets {r3} → Strategy {s3}  (all negative → uniform)")

## 4. Rock-Paper-Scissors Warmup

Before tackling CFR on a game tree, let's see regret matching in action on the simplest
possible game: **Rock-Paper-Scissors** (RPS).

RPS is a single-shot game — no tree traversal needed. Two players simultaneously choose
an action, and the payoff matrix determines who wins.

In [ ]:
# RPS payoff matrix (from row player's perspective)
#           Rock  Paper  Scissors
# Rock       0     -1      +1
# Paper     +1      0      -1
# Scissors  -1     +1       0

RPS_ACTIONS = ['R', 'P', 'S']
RPS_PAYOFF = {
    ('R', 'R'): 0, ('R', 'P'): -1, ('R', 'S'): +1,
    ('P', 'R'): +1, ('P', 'P'): 0, ('P', 'S'): -1,
    ('S', 'R'): -1, ('S', 'P'): +1, ('S', 'S'): 0,
}


def train_rps(opponent_strategy, num_iterations=1000):
    """Train a regret-matching player against a fixed opponent strategy.
    
    Args:
        opponent_strategy: dict {'R': prob, 'P': prob, 'S': prob}
    
    Returns:
        average_strategy: the time-averaged strategy (converges to best response)
        strategy_history: list of strategies over time
    """
    regret_sum = {a: 0.0 for a in RPS_ACTIONS}
    strategy_sum = {a: 0.0 for a in RPS_ACTIONS}
    strategy_history = []
    
    for t in range(num_iterations):
        # Get current strategy from regret matching
        strategy = regret_matching(regret_sum, RPS_ACTIONS)
        
        # Sample actions
        my_action = np.random.choice(RPS_ACTIONS, p=[strategy[a] for a in RPS_ACTIONS])
        opp_action = np.random.choice(RPS_ACTIONS, p=[opponent_strategy[a] for a in RPS_ACTIONS])
        
        # Compute regret for each action I could have played
        my_payoff = RPS_PAYOFF[(my_action, opp_action)]
        for a in RPS_ACTIONS:
            counterfactual_payoff = RPS_PAYOFF[(a, opp_action)]
            regret_sum[a] += counterfactual_payoff - my_payoff
        
        # Accumulate strategy for averaging
        for a in RPS_ACTIONS:
            strategy_sum[a] += strategy[a]
        
        strategy_history.append(strategy.copy())
    
    # Compute average strategy
    total = sum(strategy_sum.values())
    average_strategy = {a: strategy_sum[a] / total for a in RPS_ACTIONS}
    
    return average_strategy, strategy_history


# Test 1: against a biased opponent (40% rock, 30% paper, 30% scissors)
biased_opp = {'R': 0.4, 'P': 0.3, 'S': 0.3}
avg_strat, hist = train_rps(biased_opp, 5000)
print(f"vs Biased (40R/30P/30S):")
print(f"  Learned: R={avg_strat['R']:.3f}  P={avg_strat['P']:.3f}  S={avg_strat['S']:.3f}")
print(f"  Expected: mostly Paper (beats Rock)\n")

# Test 2: against uniform opponent → should converge to 1/3 each
uniform_opp = {'R': 1/3, 'P': 1/3, 'S': 1/3}
avg_strat2, hist2 = train_rps(uniform_opp, 5000)
print(f"vs Uniform (33/33/33):")
print(f"  Learned: R={avg_strat2['R']:.3f}  P={avg_strat2['P']:.3f}  S={avg_strat2['S']:.3f}")
print(f"  Expected: ~1/3 each (Nash equilibrium)")

# Plot strategy convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, history, title in [(axes[0], hist, 'vs Biased Opponent'), (axes[1], hist2, 'vs Uniform Opponent')]:
    # Compute running average
    for action, color in [('R', '#ef4444'), ('P', '#3b82f6'), ('S', '#22c55e')]:
        running = np.cumsum([h[action] for h in history]) / np.arange(1, len(history) + 1)
        ax.plot(running, label=action, color=color, linewidth=2)
    ax.set_title(f'RPS Regret Matching — {title}')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Average Strategy')
    ax.legend()
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 5. Counterfactual Regret Minimization (CFR)

Now we scale regret matching from a one-shot game (RPS) to a sequential game tree (Kuhn Poker).

CFR traverses the game tree recursively, computing **counterfactual values** at each information set.
The "counterfactual" part means: *what would the expected value be, assuming the player tries to reach
this information set?*

### The CFR Algorithm

For each iteration:
1. Traverse the game tree top-down
2. At each information set, use the current strategy (from regret matching)
3. Recurse for each action, computing the expected value of each action
4. Compute counterfactual regrets: "how much better would action $a$ have been?"
5. Accumulate regrets (weighted by the opponent's reach probability)
6. Update the strategy via regret matching
7. The **average** strategy (not the current one!) converges to Nash equilibrium

### Key Insight: Reach Probabilities

When we recurse through the tree, we track two **reach probabilities**:
- $\pi_1$: probability that Player 1's strategy leads to this node
- $\pi_2$: probability that Player 2's strategy leads to this node

The counterfactual regret for Player $i$ is weighted by $\pi_{-i}$ (the **opponent's** reach).
This is what makes CFR "counterfactual" — we weight by how likely the opponent is to reach
this node, not how likely we ourselves are.

In [ ]:
class CFRTrainer:
    """Counterfactual Regret Minimization for Kuhn Poker."""
    
    def __init__(self):
        self.regret_sum = defaultdict(lambda: defaultdict(float))    # regret_sum[info_set][action]
        self.strategy_sum = defaultdict(lambda: defaultdict(float))  # strategy_sum[info_set][action]
        self.info_set_actions = {}  # cache of valid actions per info set
    
    def get_strategy(self, info_set, actions):
        """Get current strategy for an info set via regret matching."""
        self.info_set_actions[info_set] = actions
        return regret_matching(self.regret_sum[info_set], actions)
    
    def cfr(self, game, reach_0, reach_1):
        """Recursive CFR traversal. Returns expected utility for Player 1.
        
        Args:
            game: current KuhnPoker state
            reach_0: probability that Player 0's strategy reaches this node
            reach_1: probability that Player 1's strategy reaches this node
        
        Returns:
            Expected value for Player 1 at this node.
        """
        # Terminal node: return payoff
        if game.is_terminal():
            return game.get_payoff()
        
        player = game.current_player
        info_set = game.get_info_set()
        actions = game.get_valid_actions()
        
        # Get current strategy from regret matching
        strategy = self.get_strategy(info_set, actions)
        
        # Compute the value of each action by recursing
        action_values = {}
        node_value = 0.0
        
        for action in actions:
            next_game = game.play(action)
            
            # Update reach probabilities
            if player == 0:
                action_values[action] = self.cfr(next_game, reach_0 * strategy[action], reach_1)
            else:
                action_values[action] = self.cfr(next_game, reach_0, reach_1 * strategy[action])
            
            node_value += strategy[action] * action_values[action]
        
        # Accumulate counterfactual regrets
        # Regret = (value of this action) - (value of the strategy we played)
        # Weighted by the OPPONENT's reach probability
        opponent_reach = reach_1 if player == 0 else reach_0
        
        for action in actions:
            regret = action_values[action] - node_value
            # For Player 1, values are already from P1's perspective.
            # For Player 0, values are also from P1's perspective, so P0's regret needs sign flip.
            if player == 0:
                self.regret_sum[info_set][action] += opponent_reach * regret
            else:
                # Player 1's perspective: their utility is -P1_utility
                self.regret_sum[info_set][action] += opponent_reach * (-regret)
        
        # Accumulate strategy sum (for computing average strategy later)
        my_reach = reach_0 if player == 0 else reach_1
        for action in actions:
            self.strategy_sum[info_set][action] += my_reach * strategy[action]
        
        return node_value
    
    def train(self, num_iterations=10000):
        """Run CFR for num_iterations, iterating over all possible deals."""
        exploitability_history = []
        
        for t in range(num_iterations):
            # Iterate over all possible card deals
            for c1 in CARDS:
                for c2 in CARDS:
                    if c1 != c2:
                        game = KuhnPoker()
                        game.cards = [c1, c2]
                        game.history = ''
                        game.current_player = 0
                        self.cfr(game, 1.0, 1.0)
            
            # Track exploitability every 100 iterations
            if (t + 1) % 100 == 0:
                exploit = self.compute_exploitability()
                exploitability_history.append((t + 1, exploit))
        
        return exploitability_history
    
    def get_average_strategy(self):
        """Compute the average strategy across all iterations.
        
        This is the strategy that converges to Nash — NOT the current strategy.
        """
        avg = {}
        for info_set, action_sums in self.strategy_sum.items():
            actions = self.info_set_actions[info_set]
            total = sum(action_sums[a] for a in actions)
            if total > 0:
                avg[info_set] = {a: action_sums[a] / total for a in actions}
            else:
                avg[info_set] = {a: 1.0 / len(actions) for a in actions}
        return avg
    
    def compute_exploitability(self):
        """Compute how exploitable the current average strategy is.
        
        Exploitability = sum of how much each player can gain by deviating.
        At Nash equilibrium, exploitability = 0.
        """
        avg_strategy = self.get_average_strategy()
        
        # Best response value for Player 0
        br_value_0 = self._best_response_value(avg_strategy, 0)
        # Best response value for Player 1  
        br_value_1 = self._best_response_value(avg_strategy, 1)
        
        # Game value of Kuhn Poker is -1/18 for Player 1
        return br_value_0 + br_value_1
    
    def _best_response_value(self, strategy, br_player):
        """Compute the value of the best response for br_player against strategy."""
        total = 0.0
        count = 0
        
        for c1 in CARDS:
            for c2 in CARDS:
                if c1 != c2:
                    game = KuhnPoker()
                    game.cards = [c1, c2]
                    game.history = ''
                    game.current_player = 0
                    total += self._br_traverse(game, strategy, br_player)
                    count += 1
        
        return total / count
    
    def _br_traverse(self, game, strategy, br_player):
        """Traverse the tree computing best response value for br_player."""
        if game.is_terminal():
            payoff = game.get_payoff()
            return payoff if br_player == 0 else -payoff
        
        player = game.current_player
        info_set = game.get_info_set()
        actions = game.get_valid_actions()
        
        if player == br_player:
            # Best response: pick the best action
            values = []
            for action in actions:
                values.append(self._br_traverse(game.play(action), strategy, br_player))
            return max(values)
        else:
            # Fixed strategy player
            strat = strategy.get(info_set, {a: 1.0/len(actions) for a in actions})
            value = 0.0
            for action in actions:
                value += strat[action] * self._br_traverse(game.play(action), strategy, br_player)
            return value

## 6. Training CFR on Kuhn Poker

Let's run CFR for 10,000 iterations and watch the strategy converge to Nash equilibrium.
We'll track **exploitability** — the total amount either player could gain by deviating.
At the Nash equilibrium, exploitability = 0.

In [ ]:
print("Training CFR on Kuhn Poker (10,000 iterations)...\n")
cfr = CFRTrainer()
exploit_history = cfr.train(num_iterations=10000)

# Plot exploitability convergence
iters, exploits = zip(*exploit_history)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(iters, exploits, color='#6366f1', linewidth=2)
ax.set_xlabel('CFR Iteration')
ax.set_ylabel('Exploitability')
ax.set_title('CFR Convergence on Kuhn Poker')
ax.axhline(y=0, color='#22c55e', linestyle='--', alpha=0.5, label='Nash (0)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Final exploitability: {exploits[-1]:.6f}")

In [ ]:
# Print the converged strategy
avg_strategy = cfr.get_average_strategy()

print("Converged Average Strategy (Nash Equilibrium Approximation):\n")
print(f"{'Info Set':<10} {'Player':<8} {'Action 1':<20} {'Action 2':<20}")
print("─" * 60)

for info_set in sorted(avg_strategy.keys()):
    actions = cfr.info_set_actions[info_set]
    player = info_sets[info_set]['player']
    strat = avg_strategy[info_set]
    
    parts = []
    for a in actions:
        label = {'b': 'bet', 'c': 'check', 'k': 'call', 'f': 'fold'}[a]
        parts.append(f"{label}: {strat[a]:.3f}")
    
    print(f"{info_set:<10} P{player+1:<7} {parts[0]:<20} {parts[1]:<20}")

## 7. Reading the Nash Equilibrium

The analytical Nash equilibrium for Kuhn Poker (Kuhn, 1950) is:

**Player 1 (first to act):**
| Card | Action | Strategy |
| :--- | :--- | :--- |
| **J** | bet/check | bet with probability $\alpha \in [0, 1/3]$, check otherwise |
| **Q** | bet/check | always check |
| **Q:cb** | call/fold | call with probability $1/3 + \alpha$, fold otherwise |
| **K** | bet/check | bet with probability $3\alpha$, check otherwise |
| **K:cb** | call/fold | always call |

**Player 2 (second to act):**
| Card | Action | Strategy |
| :--- | :--- | :--- |
| **J:b** | call/fold | always fold |
| **J:c** | bet/check | bet with probability $1/3$ |
| **Q:b** | call/fold | call with probability $1/3$ |
| **Q:c** | bet/check | always check |
| **K:b** | call/fold | always call |
| **K:c** | bet/check | bet with probability $1/3$ |

The game value is $-1/18 \approx -0.056$ for Player 1 (slight disadvantage from acting first).

Key observations:
- **King**: always call a bet (you have the best possible hand)
- **Jack**: Player 1 sometimes **bluffs** (bets with a bad hand), Player 2 always folds to a bet
- **Queen**: the tricky middle card — mixed strategies everywhere

In [ ]:
def evaluate_strategy(strategy, opponent_strategy, num_games=10000):
    """Play Kuhn Poker games with the given strategies and return average payoff for P1.
    
    strategy: dict {info_set: {action: prob}} for Player 1
    opponent_strategy: dict {info_set: {action: prob}} for Player 2
    """
    total_payoff = 0.0
    
    for _ in range(num_games):
        game = KuhnPoker().deal()
        
        while not game.is_terminal():
            info_set = game.get_info_set()
            actions = game.get_valid_actions()
            
            if game.current_player == 0:
                strat = strategy.get(info_set, {a: 1/len(actions) for a in actions})
            else:
                strat = opponent_strategy.get(info_set, {a: 1/len(actions) for a in actions})
            
            probs = [strat.get(a, 1/len(actions)) for a in actions]
            probs = [p / sum(probs) for p in probs]  # normalize
            action = np.random.choice(actions, p=probs)
            game = game.play(action)
        
        total_payoff += game.get_payoff()
    
    return total_payoff / num_games


# Test: Nash vs Nash → game value ≈ -1/18 ≈ -0.056
nash_value = evaluate_strategy(avg_strategy, avg_strategy, 50000)
print(f"Nash vs Nash:    avg P1 payoff = {nash_value:.4f}  (expected: -0.056)")

# Test: Nash vs Random → Nash should exploit random
random_strat = {info: {a: 1/len(acts) for a, acts in [(a, info_sets[info]['actions']) for a in info_sets[info]['actions']]}
                for info in info_sets}
# Simpler: just use uniform
nash_vs_random = evaluate_strategy(avg_strategy, {}, 50000)
print(f"Nash vs Random:  avg P1 payoff = {nash_vs_random:.4f}  (should be positive)")

# Test: Random vs Nash → Random should lose
random_vs_nash = evaluate_strategy({}, avg_strategy, 50000)
print(f"Random vs Nash:  avg P1 payoff = {random_vs_nash:.4f}  (should be negative)")

# Test: always-bet vs Nash
always_bet = {info: {'b': 1.0, 'c': 0.0} if 'b' in info_sets[info]['actions'] else 
              {'k': 1.0, 'f': 0.0}
              for info in info_sets if info_sets[info]['player'] == 0}
bet_vs_nash = evaluate_strategy(always_bet, avg_strategy, 50000)
print(f"AlwaysBet vs Nash: avg P1 payoff = {bet_vs_nash:.4f}  (should be ≤ -0.056)")

## 8. Bluffing Emerges from Math

The most remarkable result: **bluffing is not a human psychological trick — it emerges mathematically
from the Nash equilibrium.**

With a Jack (worst hand), the optimal strategy for Player 1 sometimes **bets** (bluffs).
Why? Because if you *never* bluff, your opponent knows that a bet always means a strong hand
and will always fold. But then bluffing becomes profitable — so you must bluff sometimes.

This is the fundamental insight of game theory: **mixed strategies arise because pure strategies
are exploitable.**

In [ ]:
# Visualize bluffing frequencies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Player 1 opening action (bet vs check)
p1_info_sets = ['J:', 'Q:', 'K:']
p1_bet_probs = [avg_strategy.get(info, {}).get('b', 0) for info in p1_info_sets]

ax = axes[0]
colors = ['#ef4444', '#f59e0b', '#22c55e']
bars = ax.bar(['Jack', 'Queen', 'King'], p1_bet_probs, color=colors, width=0.5)
ax.set_title('Player 1: Opening Bet Probability')
ax.set_ylabel('P(bet)')
ax.set_ylim(0, 1.1)
for bar, prob in zip(bars, p1_bet_probs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{prob:.2f}', ha='center', fontsize=12, fontweight='bold')

# Player 2 response to bet (call vs fold)
p2_info_sets = ['J:b', 'Q:b', 'K:b']
p2_call_probs = [avg_strategy.get(info, {}).get('k', 0) for info in p2_info_sets]

ax = axes[1]
bars = ax.bar(['Jack', 'Queen', 'King'], p2_call_probs, color=colors, width=0.5)
ax.set_title('Player 2: Call Probability (after P1 bets)')
ax.set_ylabel('P(call)')
ax.set_ylim(0, 1.1)
for bar, prob in zip(bars, p2_call_probs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
            f'{prob:.2f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("Key observations:")
print("  - Jack (worst hand): P1 sometimes BLUFFS (bets ~1/3 of the time)")
print("  - Queen (middle):    P1 usually checks, P2 calls ~1/3 of bets")  
print("  - King (best hand):  P1 often bets, P2 always calls")
print("\n  Bluffing is mathematically optimal — not a psychological trick!")

## 9. Scaling Up — From Kuhn to Texas Hold'em

Kuhn Poker has just **12 information sets**. Texas Hold'em has approximately **10^161**.

How did AI researchers bridge this gap?

| System | Year | Key Innovation |
| :--- | :---: | :--- |
| **CFR** | 2007 | Base algorithm (what we built today) |
| **Monte Carlo CFR** | 2009 | Sample the tree instead of full traversal |
| **Abstraction** | 2015 | Group similar hands together (reduce info sets) |
| **Libratus** | 2017 | Beat top humans at heads-up no-limit Hold'em |
| **Deep CFR** | 2019 | Use neural nets to approximate the strategy |
| **Pluribus** | 2019 | Beat 6 human professionals simultaneously |

The core ideas are the same as what we built:
- Regret matching to update strategies
- Counterfactual values to handle the game tree
- Average strategy convergence to Nash equilibrium

The engineering challenge is managing the astronomical number of information sets through
abstraction, sampling, and neural network function approximation.

## Recap

### Key Equations

| Component | Formula |
| :--- | :--- |
| **Regret** | $R^T(I,a) = \sum_t [v_i(I,a) - v_i(I)]$ |
| **Regret Matching** | $\sigma(I,a) = \frac{\max(R,0)}{\sum \max(R,0)}$ (or uniform if all $\leq 0$) |
| **Counterfactual Value** | $v_i(I) = \sum_a \sigma(I,a) \cdot v_i(I \rightarrow a)$ |
| **CFR Convergence** | Exploitability $\to 0$ at rate $O(1/\sqrt{T})$ |

### What We Learned
1. **Imperfect information** breaks minimax and MCTS — you can't search what you can't see
2. **Information sets** group indistinguishable states — your strategy must be the same for all states in a set
3. **Regret matching** converts cumulative regrets into a strategy: do more of what you regret not doing
4. **CFR** traverses the game tree, computing counterfactual regrets weighted by the opponent's reach
5. The **average strategy** (not the current one) converges to a Nash equilibrium
6. **Bluffing** emerges naturally from the math — it's not a psychological trick

---

### The Full Journey

From Notebook 01 to Notebook 17, we've traveled the full arc of reinforcement learning:

| Act | Notebooks | Theme |
| :--- | :---: | :--- |
| **I. Foundations** | 01-03 | MDPs, Bellman, Dynamic Programming |
| **II. Value-Based** | 04-06 | Q-Learning, DQN, Double DQN |
| **III. Policy-Based** | 07-10 | REINFORCE, Actor-Critic, PPO |
| **IV. Engineering** | 11-13 | Distributed RL, Offline RL, Model-Based RL |
| **V. LLM Alignment** | 14-15 | GRPO, DPO |
| **VI. Grandmasters** | 16-17 | AlphaZero, CFR & Poker |

From tabular Bellman equations to self-play in perfect-information games to Nash equilibria
in hidden-information games — the journey is complete.

**What you've built**: the mathematical foundations and practical implementations that power
everything from game-playing AIs to the alignment of large language models. These are not
just academic exercises — they are the building blocks of modern AI.